In [10]:
!pip install pandas numpy scikit-learn matplotlib pyarrow -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
import numpy as np
import os,json,logging,csv,time,random
from datetime import date,datetime,timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: Modular ETL Scripts and Logging")

✓ Environment ready for: Modular ETL Scripts and Logging


In [12]:
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })
df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()



✓ Dataset loaded: 10000 rows, 9 columns


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor,Laptop,North,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor,Monitor,North,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse,Keyboard,North,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub,Mouse,South,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard,Headset,West,22,285.08,2024-04-20,refunded,6271.76


In [13]:
print('=' * 60)
print('DATA PROFILE: Modular ETL Scripts and Logging')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')

for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: Modular ETL Scripts and Logging

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
  cate

In [20]:
import logging
import pandas as pd
import os,time
from datetime import datetime

#configuring dedicated logger
etl_logger = logging.getLogger('etl_pipeling')
etl_logger.setLevel(logging.INFO)

#extracting data from the sources

def extract(data = None, source = None):
    start = time.time()
    etl_logger.info('Extraction is starting')
    if data is not None:
        result = data.copy()
    elif source and os.path.exists(source):
        result = pd.read_csv(source)
    else:
        return FileNotFoundError(f'source not found : {source}')
    elapsed = time.time()
    etl_logger.info(f'EXTRACT: Loaded {len(result)} rows, {len(result.columns)} cols in {elapsed:.2f}s')
    return result

# validating the data

def validate(df,requiredcolumns=None,threshold=0.05):
    start = time.time()
    etl_logger.info('VALIDATE: Starting validation...')
    errors = []

    # we need to check required columns
    if requiredcolumns:
        missing = set(requiredcolumns)-set(df.columns)
        if missing:
            errors.append(f'Missing columns : {missing}')
    
    # then we need to check thresholds

    nulls = df.isnull().sum()
    highnulls = nulls[nulls>threshold]
    if len(highnulls)>0:
        for col,pct in highnulls.items():
            etl_logger.warning(f'VALIDATE: Column "{col}" has {pct:.1%} nulls ,threshold: {null_threshold:.1%}')
    
    # check if any duplicates are present
    if 'order_id' in df.columns:
        duplicates = df['order_id'].duplicated().sum()
        if duplicates:
            errors.append(f'{dups} duplicate order_ids found')
    
    if errors:
        for err in errors:
            etl_logger.error(f'VALIDATE FAIL: {err}')
        raise ValueError(f'Validation failed: {errors}')
    elapsed = time.time()
    etl_logger.info(f'VALIDATE: Passed all checks in {elapsed:.2f}s')
    return df

# transform - applying bussiness logic for data 

def transform(df):

    start = time.time()
    etl_logger.info('Tranformation has started')

    # standardising regional columns

    df['region'] = df['region'].str.strip().str.title()
    df['product'] = df['product'].str.strip().str.title()

    # parsing of data

    df['order_date'] = pd.to_datetime(df['order_date'])
    df['year_month'] = df['order_date'].dt.to_period('M').astype(str)

    # recalculating revenue
    df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

    # filling missing values

    df['quantity'] = df['quantity'].fillna(df['quantity'].median())

    df['etl_timestamp'] = datetime.now().isoformat()
    df['etl_batch_id'] = f'BATCH-{datetime.now().strftime("%Y%m%d%H%M%S")}'

    elapsed = time.time() - start
    etl_logger.info(f'TRANSFORM: Applied transformations in {elapsed:.2f}s')
    return df

# creating summary table : using aggregation

def aggregate(df):
    start = time.time()
    etl_logger.info("Starting aggregation")
    agg = df.groupby(['region','year_month']).agg(order_count=('order_id', 'count'),
        total_revenue=('revenue', 'sum'),
        avg_order_value=('revenue', 'mean'),
        unique_products=('product', 'nunique')).round(2).reset_index()
    elapsed = time.time()
    etl_logger.info(f'AGGREGATE: Created {len(agg)} summary rows in {elapsed:.2f}s')
    return agg 

# saving results - loading

def loading(df, agg, dest='./tmp/etl_output'):
    start = time.time()
    etl_logger.info('LOAD: Saving outputs...')
    os.makedirs(dest,exist_ok=True)
    detail_path = os.path.join(dest, 'orders_clean.csv')
    agg_path = os.path.join(dest, 'summary.csv')

    df.to_csv(detail_path,index=False)
    agg.to_csv(agg_path)

    elapsed = time.time()
    etl_logger.info(f'LOAD: Saved {len(df)} detail rows to {detail_path}')
    etl_logger.info(f'LOAD: Saved {len(agg)} summary rows to {agg_path}')
    etl_logger.info(f'LOAD: Complete in {elapsed:.2f}s')
    return detail_path, agg_path

# next we need to run the full pipeline

def runpipeline(data):
    pipeline_start = time.time()
    etl_logger.info('PIPELINE: Starting ETL run...')
    try:
        raw = extract(data = data)
        validated = validate(raw, requiredcolumns=['order_id', 'product', 'revenue'])
        transformed = transform(validated)
        summary = aggregate(transformed)
        detail_path, agg_path = loading(transformed, summary,dest='./tmp/etl_output')
        elapsed = time.time() - pipeline_start
        etl_logger.info(f'PIPELINE: SUCCESS in {elapsed:.2f}s')
        return transformed, summary
    except Exception as e:
        etl_logger.error(f'PIPELINE: FAILED — {e}')
        raise
df_transformed, df_summary = runpipeline(df)



2026-03-26 12:09:59,806 [INFO] PIPELINE: Starting ETL run...
2026-03-26 12:09:59,806 [INFO] Extraction is starting
2026-03-26 12:09:59,809 [INFO] EXTRACT: Loaded 10000 rows, 9 cols in 1774507199.81s
2026-03-26 12:09:59,810 [INFO] VALIDATE: Starting validation...
2026-03-26 12:09:59,814 [INFO] VALIDATE: Passed all checks in 1774507199.81s
2026-03-26 12:09:59,816 [INFO] Tranformation has started
2026-03-26 12:09:59,830 [INFO] TRANSFORM: Applied transformations in 0.01s
2026-03-26 12:09:59,830 [INFO] Starting aggregation
2026-03-26 12:09:59,846 [INFO] AGGREGATE: Created 48 summary rows in 1774507199.85s
2026-03-26 12:09:59,848 [INFO] LOAD: Saving outputs...
2026-03-26 12:09:59,900 [INFO] LOAD: Saved 10000 detail rows to ./tmp/etl_output\orders_clean.csv
2026-03-26 12:09:59,900 [INFO] LOAD: Saved 48 summary rows to ./tmp/etl_output\summary.csv
2026-03-26 12:09:59,900 [INFO] LOAD: Complete in 1774507199.90s
2026-03-26 12:09:59,900 [INFO] PIPELINE: SUCCESS in 0.09s


In [21]:
print('=' * 60)
print('RESULTS: Modular ETL Scripts and Logging')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: Modular ETL Scripts and Logging
Input rows: 10000
Processing complete

Revenue by Region:
region
West     32704052.11
East     31929463.06
South    30902956.65
North    30756088.37
Name: revenue, dtype: float64


In [22]:
# Cell 6: Self-Check — Modular ETL
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — Modular ETL')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — Modular ETL
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.


In [ ]:
import pandas as pd
import sqlite3
import logging
from datetime import datetime
import os

logger = logging.getLogger("invoice_etl")
logger.setLevel(logging.INFO)

def extract_csv(file_path):
    logger.info("EXTRACT: Reading CSV...")
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except:
        df = pd.read_csv(file_path, encoding='latin1')
    logger.info(f"Loaded {len(df)} rows")
    return df

def validate_data(df):
    logger.info("VALIDATE: Checking data...")
    required_cols = ['invoice_id', 'product_id', 'amount_usd', 'quantity', 'invoice_date']
    
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    if df.isnull().mean().max() > 0.05:
        raise ValueError("Too many null values")

    if (df['amount_usd'] <= 0).any():
        raise ValueError("Invalid amount")

    if (df['quantity'] <= 0).any():
        raise ValueError("Quantity must be positive")

    if df['product_id'].str.len().min() < 3:
        raise ValueError("Invalid product_id format")

    if df.duplicated(subset=['invoice_id']).any():
        raise ValueError("Duplicate invoice_id found")

    logger.info("VALIDATION PASSED")
    return df

def transform_data(df, exchange_rate=83):
    logger.info("TRANSFORM: Processing data...")
    
    df['amount_inr'] = (df['amount_usd'] * exchange_rate).round(2)
    df['product_id'] = df['product_id'].str.upper().str[:6]
    
    df['invoice_date'] = pd.to_datetime(df['invoice_date'])
    df['year_month'] = df['invoice_date'].dt.to_period('M').astype(str)
    
    df['is_high_value'] = df['amount_inr'] > 50000
    
    logger.info("TRANSFORMATION COMPLETE")
    return df

def load_to_sqlite(df, db_path='invoice_warehouse.db'):
    logger.info("LOAD: Writing to SQLite...")
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    df = df.copy()
    df['invoice_date'] = df['invoice_date'].astype(str)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS invoices (
        invoice_id TEXT PRIMARY KEY,
        product_id TEXT,
        amount_usd REAL,
        amount_inr REAL,
        quantity INTEGER,
        invoice_date TEXT,
        year_month TEXT,
        is_high_value BOOLEAN
    )
    """)

    for _, row in df.iterrows():
        cursor.execute("""
        INSERT INTO invoices VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(invoice_id) DO UPDATE SET
            product_id=excluded.product_id,
            amount_usd=excluded.amount_usd,
            amount_inr=excluded.amount_inr,
            quantity=excluded.quantity,
            invoice_date=excluded.invoice_date,
            year_month=excluded.year_month,
            is_high_value=excluded.is_high_value
        """, tuple(row))

    conn.commit()
    conn.close()
    logger.info("LOAD COMPLETE")

def run_invoice_pipeline(file_path):
    try:
        df = extract_csv(file_path)
        df = validate_data(df)
        df = transform_data(df)
        load_to_sqlite(df)
        print(" Pipeline executed successfully")
        return df
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        raise

sample_data = pd.DataFrame({
    'invoice_id': ['INV001','INV002','INV003'],
    'product_id': ['abc123','def456','ghi789'],
    'amount_usd': [100, 200, 300],
    'quantity': [1,2,3],
    'invoice_date': ['2024-01-01','2024-02-01','2024-03-01']
})

sample_file = '/tmp/sample_invoice.csv'
sample_data.to_csv(sample_file, index=False)

df_result = run_invoice_pipeline(sample_file)

df_result.head()

2026-03-26 12:38:01,739 [INFO] EXTRACT: Reading CSV...
2026-03-26 12:38:01,743 [INFO] Loaded 3 rows
2026-03-26 12:38:01,745 [INFO] VALIDATE: Checking data...
2026-03-26 12:38:01,748 [INFO] VALIDATION PASSED
2026-03-26 12:38:01,749 [INFO] TRANSFORM: Processing data...
2026-03-26 12:38:01,758 [INFO] TRANSFORMATION COMPLETE
2026-03-26 12:38:01,758 [INFO] LOAD: Writing to SQLite...
2026-03-26 12:38:01,770 [INFO] LOAD COMPLETE


✅ Pipeline executed successfully


,invoice_id,product_id,amount_usd,quantity,invoice_date,amount_inr,year_month,is_high_value
0,INV001,ABC123,100,1,2024-01-01,8300,2024-01,False
1,INV002,DEF456,200,2,2024-02-01,16600,2024-02,False
2,INV003,GHI789,300,3,2024-03-01,24900,2024-03,False
